# Gradio Intro

⚠️ **Remember to copy this notebook in your Drive and rename.**

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

**🗒️ Gradio's [Documentation](https://www.gradio.app/docs)**

## Install Gradio (and Restart)

In [ ]:
!pip install gradio --quiet

### Mount Drive

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Setup

In [ ]:
%cd /content
!rm -rf iaac_genai
!git clone https://github.com/jamesmcbennett/iaac_genai
%cd /content/iaac_genai/

In [ ]:
import sys
sys.path.append('/content/iaac_genai')

In [ ]:
!pip install -q -r requirements.txt --quiet > /dev/null 2>&1

In [ ]:
from config import Config
from utils import set_image_path, save_image, save_yml, save_svg
import torch

from diffusers.utils import load_image
from PIL import Image, ImageDraw, ImageFont
import gradio as gr
import numpy as np
import cv2
import math

## Hello World

In [ ]:
def hello(name):
	return "Hi " + name + "!"

#demo = gr.Interface(fn=hello, inputs="text", outputs="text")
demo = gr.Interface(fn=hello, inputs=gr.Textbox(lines=5, placeholder="Name Here"), outputs=gr.Textbox(lines=5))

demo.launch(share=True, debug=True)

## MaCAD Groups

In [ ]:
def creative_image_generation(first_name, mode, group_num):
    if mode == "Sync":
        group = "S" + str(group_num)
    else:
        group = "AS" + str(group_num)
    return "Hello " + first_name + " of GenAI " + str(group)

demo = gr.Interface(fn=creative_image_generation,
					inputs=[
                        gr.Textbox(lines=2, placeholder="Name Here", label="First Name"),
                        gr.Radio(choices=["Sync", "Async"], label="Mode"),
                        gr.Slider(0,12, step=1, label="Group Number")
                    ],
					outputs=[
                        gr.Text(label="IAAC Gen AI", lines=10),
                    ])

demo.launch(share=True, debug=True)

## Show Examples

In [ ]:
def show_examples(name):
    return f"Hello. {name} is a sync student."

demo = gr.Interface(fn=show_examples, inputs="text", outputs="text", examples=[["Elena"], ["Eleni"], ["Tue Minh"], ["Shuai"], ["Symon"], ["Andrea"], ["Aditya"], ["Eva"], ["Eduardo"]])

demo.launch(share=True, debug=True)

## Simple Calculator

In [ ]:
def calculator(num1, operation, total):
    if operation == "add":
        total += num1
    elif operation == "subtract":
        total -= num1
    elif operation == "multiply":
        total *= num1
    elif operation == "divide":
        if num1 != 0:
            total /= num1
        else:
            return "Error: Divide by zero", total
    return total, total

with gr.Blocks() as demo:
    state = gr.State(0)

    gr.Markdown("### Simple Calculator")

    with gr.Row():
        num = gr.Number(label="Enter a number")
        op = gr.Radio(["add", "subtract", "multiply", "divide"], label="Operation")

    calc_btn = gr.Button("Calculate")

    output = gr.Number(label="Current Total")

    calc_btn.click(fn=calculator, inputs=[num, op, state], outputs=[output, state])

demo.launch(share=True, debug=True)

## Sepia Image

In [ ]:
def sepia(image):
    image = image.astype(np.float32)

    sepia_filter = np.array([
        [0.393, 0.769, 0.189], # R_Channel
        [0.349, 0.686, 0.168], # G_Channel
        [0.272, 0.534, 0.131]  # B_Channel
    ])

    sepia_image = image @ sepia_filter.T
    sepia_image = np.clip(sepia_image, 0, 255).astype(np.uint8)

    return sepia_image

demo = gr.Interface(
    fn=sepia,
        inputs=gr.Image(type="numpy", image_mode="RGB"),
        #inputs=gr.Image(value=input_image, type="numpy", image_mode="RGB"), #give a default image
        outputs=gr.Image(type="numpy")
    )

demo.launch(share=True, debug=True)

## Multiple Filters

In [ ]:
def sepia(image):
    image = image.astype(np.float32)
    sepia_filter = np.array([
        [0.393, 0.769, 0.189],
        [0.349, 0.686, 0.168],
        [0.272, 0.534, 0.131]
    ])
    sepia_image = image @ sepia_filter.T
    return np.clip(sepia_image, 0, 255).astype(np.uint8)

def grayscale(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

def invert(image):
    return 255 - image

def warm(image):
    warm_filter = np.array([1.1, 1.0, 0.9])
    warm_image = image.astype(np.float32) * warm_filter
    return np.clip(warm_image, 0, 255).astype(np.uint8)

def cool(image):
    cool_filter = np.array([0.9, 1.0, 1.1])
    cool_image = image.astype(np.float32) * cool_filter
    return np.clip(cool_image, 0, 255).astype(np.uint8)

def sketch(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    inv = 255 - gray
    blur = cv2.GaussianBlur(inv, (21, 21), 0)
    sketch = cv2.divide(gray, 255 - blur, scale=256)
    return cv2.cvtColor(sketch, cv2.COLOR_GRAY2RGB)

def emboss(image):
    kernel = np.array([[ -2, -1, 0],
                       [ -1,  1, 1],
                       [  0,  1, 2]])
    embossed = cv2.filter2D(image, -1, kernel) + 128
    return np.clip(embossed, 0, 255).astype(np.uint8)

def sharpen(image):
    kernel = np.array([[ 0, -1, 0],
                       [-1,  5,-1],
                       [ 0, -1, 0]])
    sharpened = cv2.filter2D(image, -1, kernel)
    return np.clip(sharpened, 0, 255).astype(np.uint8)

def blur(image):
    return cv2.GaussianBlur(image, (11, 11), 0)

def cartoon(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    blur = cv2.medianBlur(gray, 7)
    edges = cv2.adaptiveThreshold(blur, 255,
                                   cv2.ADAPTIVE_THRESH_MEAN_C,
                                   cv2.THRESH_BINARY, 9, 10)
    color = cv2.bilateralFilter(image, 9, 300, 300)
    cartoon = cv2.bitwise_and(color, color, mask=edges)
    return cartoon

def posterize(image):
    levels = 4
    factor = 256 // levels
    return (image // factor * factor).astype(np.uint8)

def brighten(image):
    return np.clip(image.astype(np.float32) + 40, 0, 255).astype(np.uint8)

filters = {
    "Sepia": sepia,
    "Grayscale": grayscale,
    "Invert": invert,
    "Warm": warm,
    "Cool": cool,
    "Sketch": sketch,
    "Emboss": emboss,
    "Sharpen": sharpen,
    "Blur": blur,
    "Cartoon": cartoon,
    "Posterize": posterize,
    "Brighten": brighten
}

def apply_filter(image, filter_type):
    return filters[filter_type](image)

demo = gr.Interface(
    fn=apply_filter,
    inputs=[
        gr.Image(type="numpy", image_mode="RGB"),
        # gr.Dropdown(choices=list(filters.keys()), label="Filter Type", value="Sepia")
        gr.Radio(choices=list(filters.keys()), label="Filter Type", value="Sepia")

    ],
    outputs=gr.Image(type="numpy"),
)

# with gr.Blocks() as demo:
#     with gr.Row():
#         img_input = gr.Image(type="numpy", image_mode="RGB")
#         filter_select = gr.Radio(choices=list(filters.keys()), value="Sepia", label="Filter")
#     img_output = gr.Image(type="numpy")
#     btn = gr.Button("Apply Filter")
#     btn.click(fn=apply_filter, inputs=[img_input, filter_select], outputs=img_output)


demo.launch(share=True, prevent_thread_lock=True)


## Create Mask in Gradio Sketch

In [ ]:
WIDTH = 1024
HEIGHT = 1024
INPUT_IMG = "/content/drive/MyDrive/iaac_genai/workflows/03_FINETUNING/02_FLUX.1/dataset_FLUX.1/bof_aar_lacha_02.png"

input_image = load_image(INPUT_IMG).resize((WIDTH, HEIGHT))
input_image

In [ ]:
def create_mask(editor_value):
    try:
        if isinstance(editor_value, dict) and isinstance(editor_value.get("layers", [None])[0], Image.Image):
            img = editor_value["layers"][0].convert("RGBA")
            alpha = img.getchannel("A")
            mask = alpha.point(lambda x: 255 if x > 0 else 0, mode="1").convert("RGB")
            mask.save("image_mask.png")
            return "✅ Mask saved as image_mask.png", mask
        else:
            return "❌ Invalid input format.", None
    except Exception as e:
        return f"❌ Error: {e}", None

with gr.Blocks() as demo:
    with gr.Row():
        sketch = gr.ImageEditor(value=input_image, label="Draw Mask", type="pil", height=500)
        mask_display = gr.Image(label="Mask Preview", visible=False, height=500)
    save_btn = gr.Button("Save Mask")
    status = gr.Textbox(label="Status", lines=1)

    def on_save(editor_value):
        msg, mask = create_mask(editor_value)
        return msg, gr.update(value=mask, visible=mask is not None)

    save_btn.click(fn=on_save, inputs=sketch, outputs=[status, mask_display])

demo.launch(share=True, prevent_thread_lock=True)

In [ ]:
# Display mask or load mask from image

USE_GRADIO_MASK = True

if USE_GRADIO_MASK:
    mask_path = "image_mask.png"
else:
    mask_path = "/content/drive/MyDrive/LoRA/dataset/bofill_0001_mask.jpg"

input_mask = Image.open(mask_path).resize((WIDTH, HEIGHT)).convert("L")
input_mask = input_mask.point(lambda x: 255 if x > 128 else 0, mode="1").convert("RGB")
input_mask

## Comic Speech Bubbles - .change() event

In [ ]:
import os

# Valid fonts
font_paths = {
    "DejaVuSans": "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "LiberationSans": "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
}
valid_fonts = {k: v for k, v in font_paths.items() if os.path.exists(v)}
default_font_name = list(valid_fonts.keys())[0] if valid_fonts else None

def draw_speech_bubble(image, text, x_pct, y_pct, width_pct, height_pct, radius, font_size, font_name, angle_deg, stretch):
    if image is None:
        image = Image.new("RGB", (512, 512), "white")
    else:
        image = image.convert("RGB")

    W, H = image.size

    # Convert percentages to absolute pixel values
    x = int((x_pct / 100) * W)
    y = int((y_pct / 100) * H)
    width = int((width_pct / 100) * W)
    height = int((height_pct / 100) * H)

    draw = ImageDraw.Draw(image)

    # Draw speech bubble (black, no border)
    bubble_box = (x, y, x + width, y + height)
    draw.rounded_rectangle(bubble_box, radius=radius, fill="black")

    # Tail origin: center of bottom edge of the bubble
    tail_base_x = x + width // 2
    tail_base_y = y + height

    # Tail tip based on angle and stretch
    angle_rad = math.radians(angle_deg)
    tip_x = tail_base_x + stretch * math.cos(angle_rad)
    tip_y = tail_base_y + stretch * math.sin(angle_rad)

    tail_half_width = 10
    tail = [
        (tail_base_x - tail_half_width, tail_base_y),
        (tip_x, tip_y),
        (tail_base_x + tail_half_width, tail_base_y),
    ]
    draw.polygon(tail, fill="black")

    # Load font
    font_path = valid_fonts.get(font_name, None)
    try:
        font = ImageFont.truetype(font_path, font_size)
    except:
        font = ImageFont.load_default()

    # Centered text
    text_bbox = draw.textbbox((0, 0), text, font=font)
    text_width = text_bbox[2] - text_bbox[0]
    text_height = text_bbox[3] - text_bbox[1]
    text_x = x + (width - text_width) // 2
    text_y = y + (height - text_height) // 2
    draw.text((text_x, text_y), text, font=font, fill="white")

    return image

# Themes
# gr.themes.Default()
# gr.themes.Soft()
# gr.themes.Monochrome()
# gr.themes.Glass()
# gr.themes.Base()
# gr.themes.Ocean()

# Gradio Interface
with gr.Blocks(gr.themes.Glass()) as demo:
    gr.Markdown("## 💬 Speech Bubble Editor")

    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(type="pil", label="Upload Image", height=400)
        with gr.Column(scale=1):
            output_img = gr.Image(label="Preview", height=400)

    with gr.Row():
        text = gr.Textbox(label="Text", value="Hello!", scale=3)
        font = gr.Dropdown(choices=list(valid_fonts.keys()), value=default_font_name, label="Font", scale=1)
        font_size = gr.Slider(10, 60, step=1, value=24, label="Font Size", scale=1)

    gr.Markdown("#### 📐 Position & Size")
    with gr.Row():
        x = gr.Slider(0, 100, step=1, value=64, label="X (%)")
        y = gr.Slider(0, 100, step=1, value=43, label="Y (%)")
        width = gr.Slider(1, 100, step=1, value=27, label="Width (%)")
        height = gr.Slider(1, 100, step=1, value=8, label="Height (%)")

    gr.Markdown("#### 🗨️ Bubble & Tail")
    with gr.Row():
        radius = gr.Slider(0, 100, step=1, value=10, label="Corner Radius (px)")
        angle = gr.Slider(0, 360, step=1, value=150, label="Tail Angle (°)")
        stretch = gr.Slider(10, 300, step=1, value=40, label="Tail Length (px)")

    inputs = [input_img, text, x, y, width, height, radius, font_size, font, angle, stretch]
    for control in inputs:
        control.change(fn=draw_speech_bubble, inputs=inputs, outputs=output_img)

demo.launch(share=True, prevent_thread_lock=True)
